# 05 - Contrefactuels et Archétypes

**Étape 5 du pipeline AntakIA** : Génération d'explications "What-If".

Ce notebook démontre :
1. Génération de contrefactuels avec DiCE
2. Identification des archétypes par tesselle
3. Explication actionnable des décisions

In [ ]:
# Configuration
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("✅ Imports de base chargés")

In [ ]:
# Vérification DiCE
try:
    import dice_ml
    from dice_ml import Dice
    print(f"✅ DiCE disponible")
    DICE_AVAILABLE = True
except ImportError:
    print("⚠️ DiCE non disponible")
    print("   Installation : pip install dice-ml")
    DICE_AVAILABLE = False

In [ ]:
# Import modules AntakIA
try:
    from antakia.tessellation.counterfactuals import (
        generate_counterfactuals,
        find_archetype_and_counterfactual,
        Counterfactual,
        CounterfactualResult
    )
    print("✅ Modules AntakIA counterfactuals chargés")
    ANTAKIA_CF_AVAILABLE = True
except ImportError as e:
    print(f"⚠️ Modules AntakIA non disponibles: {e}")
    ANTAKIA_CF_AVAILABLE = False

## 1. Préparation des Données

In [ ]:
# Dataset Breast Cancer (classification binaire)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# Simplifier les noms de features
X.columns = [c.replace(' ', '_') for c in X.columns]

# Sélectionner les 10 features principales
top_features = ['mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area',
                'mean_smoothness', 'mean_compactness', 'mean_concavity',
                'mean_concave_points', 'mean_symmetry', 'mean_fractal_dimension']
X = X[top_features]

# Noms métier
FEATURE_ALIASES = {
    'mean_radius': 'Rayon moyen',
    'mean_texture': 'Texture moyenne',
    'mean_perimeter': 'Périmètre moyen',
    'mean_area': 'Surface moyenne',
    'mean_smoothness': 'Lissage moyen',
    'mean_compactness': 'Compacité moyenne',
    'mean_concavity': 'Concavité moyenne',
    'mean_concave_points': 'Points concaves moyens',
    'mean_symmetry': 'Symétrie moyenne',
    'mean_fractal_dimension': 'Dimension fractale moyenne'
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"📊 Données: {X.shape[0]} observations, {X.shape[1]} features")
print(f"   Classes: {dict(y.value_counts())}")
print(f"   0=Malin, 1=Bénin")

In [ ]:
# Entraîner un modèle
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"✅ Modèle entraîné (Accuracy = {accuracy:.3f})")

## 2. Génération de Contrefactuels avec DiCE

In [ ]:
if DICE_AVAILABLE:
    # Préparer DiCE
    # Créer le DataFrame avec la target
    df_train = X_train.copy()
    df_train['target'] = y_train.values
    
    # Définir les features
    continuous_features = list(X.columns)
    
    # Créer le Data object DiCE
    d = dice_ml.Data(
        dataframe=df_train,
        continuous_features=continuous_features,
        outcome_name='target'
    )
    
    # Créer le Model object DiCE
    m = dice_ml.Model(model=model, backend='sklearn')
    
    # Créer l'explainer DiCE
    exp = Dice(d, m, method='random')  # ou 'genetic', 'kdtree'
    
    print("✅ DiCE Explainer initialisé")

In [ ]:
if DICE_AVAILABLE:
    # Sélectionner un cas à expliquer (prédit malin)
    # Trouver un cas prédit comme 0 (malin)
    y_pred = model.predict(X_test)
    malignant_indices = X_test[y_pred == 0].index[:5]
    
    if len(malignant_indices) > 0:
        test_idx = malignant_indices[0]
        query_instance = X_test.loc[[test_idx]]
        
        print(f"📍 Cas #{test_idx}")
        print(f"   Prédiction actuelle: MALIN (0)")
        print(f"   Probabilité: {model.predict_proba(query_instance)[0]}")
    else:
        print("⚠️ Aucun cas prédit comme malin trouvé")

In [ ]:
if DICE_AVAILABLE and len(malignant_indices) > 0:
    print("🔄 Génération de contrefactuels...")
    print("   (Quelles modifications pour passer à BÉNIN ?)\n")
    
    # Générer 3 contrefactuels
    dice_exp = exp.generate_counterfactuals(
        query_instance,
        total_CFs=3,
        desired_class="opposite",
        features_to_vary="all"
    )
    
    # Afficher les résultats
    dice_exp.visualize_as_dataframe(show_only_changes=True)

In [ ]:
if DICE_AVAILABLE and len(malignant_indices) > 0:
    # Analyser les changements
    original = query_instance.iloc[0]
    cf_df = dice_exp.cf_examples_list[0].final_cfs_df
    
    print("📊 Analyse des modifications suggérées:\n")
    
    for cf_idx in range(len(cf_df)):
        cf = cf_df.iloc[cf_idx]
        print(f"--- Contrefactuel {cf_idx + 1} ---")
        
        changes = []
        for col in X.columns:
            orig_val = original[col]
            cf_val = cf[col]
            if abs(orig_val - cf_val) > 0.01:
                alias = FEATURE_ALIASES.get(col, col)
                pct_change = ((cf_val - orig_val) / orig_val) * 100
                direction = "↑" if pct_change > 0 else "↓"
                changes.append({
                    'feature': alias,
                    'original': orig_val,
                    'target': cf_val,
                    'change_pct': pct_change,
                    'direction': direction
                })
        
        for c in sorted(changes, key=lambda x: abs(x['change_pct']), reverse=True):
            print(f"  • {c['feature']}: {c['original']:.2f} → {c['target']:.2f} ({c['direction']}{abs(c['change_pct']):.1f}%)")
        print()

## 3. Identification des Archétypes

In [ ]:
# Méthode simple : point le plus proche du centroïde
def find_archetype(X_cluster: pd.DataFrame, method: str = 'centroid') -> int:
    """
    Trouve l'archétype (point le plus représentatif) d'un cluster.
    
    Parameters
    ----------
    X_cluster : pd.DataFrame
        Données du cluster
    method : str
        'centroid' : point le plus proche du centre
        'medoid' : point minimisant la distance aux autres
    
    Returns
    -------
    int : Index de l'archétype
    """
    if method == 'centroid':
        centroid = X_cluster.mean()
        distances = np.sqrt(((X_cluster - centroid) ** 2).sum(axis=1))
        return distances.idxmin()
    
    elif method == 'medoid':
        from sklearn.metrics import pairwise_distances
        dist_matrix = pairwise_distances(X_cluster)
        medoid_idx = dist_matrix.sum(axis=1).argmin()
        return X_cluster.index[medoid_idx]
    
    else:
        raise ValueError(f"Unknown method: {method}")

print("✅ Fonction find_archetype définie")

In [ ]:
# Simuler des tesselles (clustering simple)
from sklearn.cluster import KMeans

# Standardiser pour le clustering
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

# Clustering
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

print(f"📊 {n_clusters} tesselles créées:")
for i in range(n_clusters):
    mask = cluster_labels == i
    n_points = mask.sum()
    pct_benign = y[mask].mean() * 100
    print(f"   Tesselle {i}: {n_points} points, {pct_benign:.1f}% bénin")

In [ ]:
# Trouver l'archétype de chaque tesselle
archetypes = {}

print("📍 Archétypes par tesselle:\n")

for i in range(n_clusters):
    mask = cluster_labels == i
    X_cluster = X[mask]
    
    archetype_idx = find_archetype(X_cluster)
    archetypes[i] = archetype_idx
    
    archetype_x = X.loc[archetype_idx]
    archetype_pred = model.predict([archetype_x])[0]
    archetype_proba = model.predict_proba([archetype_x])[0]
    
    print(f"Tesselle {i} - Archétype #{archetype_idx}")
    print(f"  Prédiction: {'Bénin' if archetype_pred == 1 else 'Malin'}")
    print(f"  Probabilité: {archetype_proba}")
    print(f"  Top features:")
    for feat in top_features[:3]:
        alias = FEATURE_ALIASES.get(feat, feat)
        print(f"    • {alias}: {archetype_x[feat]:.2f}")
    print()

## 4. Comparaison Point vs Archétype

In [ ]:
def compare_to_archetype(point_idx: int, X: pd.DataFrame, 
                         cluster_labels: np.ndarray, archetypes: dict,
                         aliases: dict = None) -> pd.DataFrame:
    """
    Compare un point à l'archétype de sa tesselle.
    """
    aliases = aliases or {}
    
    cluster = cluster_labels[X.index.get_loc(point_idx)]
    archetype_idx = archetypes[cluster]
    
    point = X.loc[point_idx]
    archetype = X.loc[archetype_idx]
    
    comparison = pd.DataFrame({
        'feature': X.columns,
        'alias': [aliases.get(f, f) for f in X.columns],
        'point': point.values,
        'archetype': archetype.values,
        'diff': point.values - archetype.values,
        'diff_pct': ((point.values - archetype.values) / archetype.values) * 100
    })
    
    return comparison.sort_values('diff_pct', key=abs, ascending=False)

print("✅ Fonction compare_to_archetype définie")

In [ ]:
# Comparer un cas au hasard
random_idx = X.index[42]
comparison = compare_to_archetype(random_idx, X, cluster_labels, archetypes, FEATURE_ALIASES)

cluster = cluster_labels[X.index.get_loc(random_idx)]
point_pred = model.predict([X.loc[random_idx]])[0]
archetype_pred = model.predict([X.loc[archetypes[cluster]]])[0]

print(f"📊 Comparaison Point #{random_idx} vs Archétype (Tesselle {cluster})")
print(f"   Point prédit: {'Bénin' if point_pred == 1 else 'Malin'}")
print(f"   Archétype prédit: {'Bénin' if archetype_pred == 1 else 'Malin'}")
print(f"\n   Différences majeures:")
print(comparison.head(5).to_string(index=False))

## 5. Visualisation

In [ ]:
# PCA pour visualisation
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 5))

# Plot 1: Par classe
plt.subplot(1, 2, 1)
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='coolwarm', alpha=0.6)
plt.colorbar(scatter, label='Classe (0=Malin, 1=Bénin)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Distribution par Classe')

# Plot 2: Par tesselle avec archétypes
plt.subplot(1, 2, 2)
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='tab10', alpha=0.6)

# Marquer les archétypes
for cluster_id, arch_idx in archetypes.items():
    arch_loc = X.index.get_loc(arch_idx)
    plt.scatter(X_pca[arch_loc, 0], X_pca[arch_loc, 1], 
                marker='*', s=300, c='black', edgecolors='white', linewidths=2,
                label=f'Archétype T{cluster_id}' if cluster_id == 0 else '')

plt.colorbar(scatter, label='Tesselle')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Tesselles et Archétypes (★)')

plt.tight_layout()
plt.show()

## 6. Résumé

In [ ]:
print("="*60)
print("📋 RÉSUMÉ - Étape 5 : Contrefactuels et Archétypes")
print("="*60)
print(f"""
✅ DiCE disponible: {DICE_AVAILABLE}
   → Génération de contrefactuels "What-If"
   → Modifications minimales pour changer la décision

✅ Archétypes identifiés:
   → {n_clusters} tesselles analysées
   → Point représentatif (proche du centroïde) par tesselle

✅ Comparaison point vs archétype:
   → Identification des différences significatives
   → Base pour explications contextualisées

🔜 Étape suivante: 06_llm_explanation_demo.ipynb
   → Explications en langage naturel
   → Rapport exécutif, décisions, chat
""")